In [1]:
url = [ 
        "https://www.motortrend.com/" ,
        "https://www.autonews.com/",
        "https://www.spglobal.com/automotive-insights/en",
        "https://www.automotiveworld.com/"
        ]

In [1]:
from __future__ import annotations

import csv
import gzip
import xml.etree.ElementTree as ET
from pathlib import Path

import requests
from openpyxl import Workbook

# -----------------------------
# CONFIG
# -----------------------------
INPUT_DIR = Path(".")  # put your 4 XML files here or change this path

INPUT_FILES = [
    "sitemap.xml",
    "sitemap_index.xml",
    "download.xml",
    "XML Sitemap.xml",
]

OUTPUT_DIR = INPUT_DIR / "sitemap_outputs"
OUTPUT_DIR.mkdir(exist_ok=True)

EXCEL_PATH = OUTPUT_DIR / "all_sitemaps.xlsx"
EXCEL_MAX_DATA_ROWS = 1_048_575  # Excel max rows is 1,048,576 including header
REQUEST_TIMEOUT = 60
HEADERS = {
    "User-Agent": (
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) Chrome/136.0 Safari/537.36"
    )
}


# -----------------------------
# XML HELPERS
# -----------------------------
def strip_ns(tag: str) -> str:
    return tag.split("}", 1)[-1] if "}" in tag else tag


def get_ns(root: ET.Element) -> dict[str, str]:
    if root.tag.startswith("{"):
        return {"sm": root.tag.split("}", 1)[0][1:]}
    return {}


def get_child_text(parent: ET.Element, child_name: str, ns: dict[str, str]) -> str:
    if ns:
        child = parent.find(f"sm:{child_name}", ns)
    else:
        child = parent.find(child_name)
    return (child.text or "").strip() if child is not None and child.text else ""


def parse_xml_bytes(content: bytes, source_name: str = "") -> ET.Element:
    maybe_gz = source_name.lower().endswith(".gz") or content[:2] == b"\x1f\x8b"
    if maybe_gz:
        try:
            content = gzip.decompress(content)
        except OSError:
            # If the server already sent decompressed content, keep going.
            pass

    text = content.decode("utf-8-sig", errors="replace").strip()
    return ET.fromstring(text)


# -----------------------------
# NETWORK
# -----------------------------
def fetch_remote_root(url: str, session: requests.Session) -> ET.Element:
    response = session.get(url, headers=HEADERS, timeout=REQUEST_TIMEOUT)
    response.raise_for_status()
    return parse_xml_bytes(response.content, source_name=url)


# -----------------------------
# OUTPUT HELPERS
# -----------------------------
def safe_sheet_name(name: str) -> str:
    invalid = ['\\', '/', '*', '[', ']', ':', '?']
    for ch in invalid:
        name = name.replace(ch, "_")
    return name[:31]


def safe_file_stem(name: str) -> str:
    keep = []
    for ch in Path(name).stem:
        keep.append(ch if ch.isalnum() or ch in ("-", "_") else "_")
    return "".join(keep).strip("_") or "output"


def write_row(
    link: str,
    lastmod: str,
    csv_writer: csv.DictWriter,
    ws,
    stats: dict,
) -> None:
    csv_writer.writerow({"link": link, "lastmod": lastmod})
    stats["csv_rows"] += 1

    if ws is not None:
        if stats["excel_rows"] < EXCEL_MAX_DATA_ROWS:
            ws.append([link, lastmod])
            stats["excel_rows"] += 1
        else:
            stats["excel_truncated"] = True


# -----------------------------
# CORE RECURSIVE WALKER
# -----------------------------
def walk_root(
    root: ET.Element,
    session: requests.Session,
    csv_writer: csv.DictWriter,
    ws,
    stats: dict,
    visited_sitemaps: set[str],
) -> None:
    ns = get_ns(root)
    root_type = strip_ns(root.tag)

    if root_type == "urlset":
        url_nodes = root.findall("./sm:url", ns) if ns else root.findall("./url")
        for url_node in url_nodes:
            link = get_child_text(url_node, "loc", ns)
            lastmod = get_child_text(url_node, "lastmod", ns)
            if link:
                write_row(link, lastmod, csv_writer, ws, stats)
        return

    if root_type == "sitemapindex":
        sitemap_nodes = root.findall("./sm:sitemap", ns) if ns else root.findall("./sitemap")
        for sitemap_node in sitemap_nodes:
            child_loc = get_child_text(sitemap_node, "loc", ns)
            if not child_loc or child_loc in visited_sitemaps:
                continue

            visited_sitemaps.add(child_loc)

            try:
                child_root = fetch_remote_root(child_loc, session)
                walk_root(child_root, session, csv_writer, ws, stats, visited_sitemaps)
            except Exception as exc:
                print(f"[WARN] Could not expand child sitemap: {child_loc} -> {exc}")
        return

    raise ValueError(f"Unsupported root tag: {root.tag}")


# -----------------------------
# FILE PROCESSOR
# -----------------------------
def process_input_file(file_path: Path, workbook: Workbook, session: requests.Session) -> None:
    stem = safe_file_stem(file_path.name)
    csv_path = OUTPUT_DIR / f"{stem}.csv"

    ws = workbook.create_sheet(title=safe_sheet_name(stem))
    ws.append(["link", "lastmod"])

    stats = {
        "csv_rows": 0,
        "excel_rows": 0,
        "excel_truncated": False,
    }

    visited_sitemaps: set[str] = set()
    root = parse_xml_bytes(file_path.read_bytes(), source_name=file_path.name)

    with csv_path.open("w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=["link", "lastmod"])
        writer.writeheader()
        walk_root(root, session, writer, ws, stats, visited_sitemaps)

    print(f"[DONE] {file_path.name} -> {csv_path} | rows={stats['csv_rows']}")
    if stats["excel_truncated"]:
        print(
            f"[WARN] Excel sheet for {file_path.name} was truncated at "
            f"{EXCEL_MAX_DATA_ROWS} data rows. Use the CSV as the full output."
        )


# -----------------------------
# MAIN
# -----------------------------
def main() -> None:
    workbook = Workbook()
    workbook.remove(workbook.active)

    with requests.Session() as session:
        for file_name in INPUT_FILES:
            file_path = INPUT_DIR / file_name
            if not file_path.exists():
                print(f"[SKIP] File not found: {file_name}")
                continue

            try:
                process_input_file(file_path, workbook, session)
            except Exception as exc:
                print(f"[ERROR] Failed to process {file_name}: {exc}")

    workbook.save(EXCEL_PATH)
    print(f"[DONE] Excel workbook saved to: {EXCEL_PATH}")


if __name__ == "__main__":
    main()

c:\Users\user\anaconda3\envs\compute\Lib\site-packages\requests\__init__.py:113: RequestsDependencyWarning: urllib3 (2.6.3) or chardet (7.3.0)/charset_normalizer (3.4.4) doesn't match a supported version!
  warnings.warn(


[DONE] sitemap.xml -> sitemap_outputs\sitemap.csv | rows=447
[DONE] sitemap_index.xml -> sitemap_outputs\sitemap_index.csv | rows=78739
[SKIP] File not found: download.xml
[DONE] XML Sitemap.xml -> sitemap_outputs\XML_Sitemap.csv | rows=110988
[DONE] Excel workbook saved to: sitemap_outputs\all_sitemaps.xlsx
